In [1]:
 --Proposition 1: Compare each employee’s current rate to their previous rate
use AdventureWorks2019
go
SELECT 
    BusinessEntityID,
    RateChangeDate,
    Rate,
    LAG(Rate) OVER (PARTITION BY BusinessEntityID ORDER BY RateChangeDate) AS PrevRate,
    Rate - LAG(Rate) OVER (PARTITION BY BusinessEntityID ORDER BY RateChangeDate) AS RateDifference
FROM HumanResources.EmployeePayHistory;

Commands completed successfully.

(316 rows affected)

Total execution time: 00:00:00.047

BusinessEntityID,RateChangeDate,Rate,PrevRate,RateDifference
1,2009-01-14 00:00:00.000,125.50,NULL,NULL
2,2008-01-31 00:00:00.000,63.4615,NULL,NULL
3,2007-11-11 00:00:00.000,43.2692,NULL,NULL
4,2007-12-05 00:00:00.000,8.62,NULL,NULL
4,2010-05-31 00:00:00.000,23.72,8.62,15.10
4,2011-12-15 00:00:00.000,29.8462,23.72,6.1262
5,2008-01-06 00:00:00.000,32.6923,NULL,NULL
6,2008-01-24 00:00:00.000,32.6923,NULL,NULL
7,2009-02-08 00:00:00.000,50.4808,NULL,NULL
8,2008-12-29 00:00:00.000,40.8654,NULL,NULL


In [4]:
--2: Rank employees by salary
USE AdventureWorks2019;
GO

SELECT 
    BusinessEntityID,
    Rate,
    RANK() OVER (ORDER BY Rate DESC) AS SalaryRank
FROM HumanResources.EmployeePayHistory;

Commands completed successfully.

(316 rows affected)

Total execution time: 00:00:00.045

BusinessEntityID,Rate,SalaryRank
1,125.50,1
25,84.1346,2
273,72.1154,3
2,63.4615,4
234,60.0962,5
263,50.4808,6
7,50.4808,6
234,48.5577,8
285,48.101,9
274,48.101,9


In [5]:
--3:  Running total of list prices per product
USE AdventureWorks2019;
GO

SELECT 
    ProductID,
    StartDate,
    ListPrice,
    SUM(ListPrice) OVER (PARTITION BY ProductID ORDER BY StartDate) AS RunningPriceTotal
FROM Production.ProductListPriceHistory;

Commands completed successfully.

(395 rows affected)

Total execution time: 00:00:00.045

ProductID,StartDate,ListPrice,RunningPriceTotal
707,2011-05-31 00:00:00.000,33.6442,33.6442
707,2012-05-30 00:00:00.000,33.6442,67.2884
707,2013-05-30 00:00:00.000,34.99,102.2784
708,2011-05-31 00:00:00.000,33.6442,33.6442
708,2012-05-30 00:00:00.000,33.6442,67.2884
708,2013-05-30 00:00:00.000,34.99,102.2784
709,2011-05-31 00:00:00.000,9.50,9.50
710,2011-05-31 00:00:00.000,9.50,9.50
711,2011-05-31 00:00:00.000,33.6442,33.6442
711,2012-05-30 00:00:00.000,33.6442,67.2884


In [8]:
--4:  ROLLUP on country and state for customer totals
USE AdventureWorks2019;
GO

SELECT 
    CountryRegionCode,
    Name AS StateProvince,
    COUNT(*) AS TotalCustomers
FROM Person.StateProvince
GROUP BY ROLLUP (CountryRegionCode, Name);

Commands completed successfully.

(194 rows affected)

Total execution time: 00:00:00.069

CountryRegionCode,StateProvince,TotalCustomers
AS,American Samoa,1
AS,NULL,1
AU,New South Wales,1
AU,Queensland,1
AU,South Australia,1
AU,Tasmania,1
AU,Victoria,1
AU,NULL,5
CA,Alberta,1
CA,British Columbia,1


In [11]:
---5 :Group orders into 4 quartiles based on TotalDue
USE AdventureWorks2019
GO
SELECT 
  SalesOrderID,
  TotalDue,
  NTILE(4) OVER(ORDER BY TotalDue DESC) AS Quartile
FROM Sales.SalesOrderHeader;

Commands completed successfully.

(31465 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:00.307

SalesOrderID,TotalDue,Quartile
51131,187487.825,1
55282,182018.6272,1
46616,170512.6689,1
46981,166537.0808,1
47395,165028.7482,1
47369,158056.5449,1
47355,145741.8553,1
51822,145454.366,1
44518,142312.2199,1
51858,140042.1209,1


In [13]:
--6 :Find first and last price for each product
USE AdventureWorks2019
GO
SELECT 
  ProductID,
  ListPrice,
  StartDate,
  FIRST_VALUE(ListPrice) OVER(PARTITION BY ProductID ORDER BY StartDate) AS FirstPrice,
  LAST_VALUE(ListPrice) OVER(PARTITION BY ProductID ORDER BY StartDate 
      ROWS BETWEEN CURRENT ROW AND UNBOUNDED FOLLOWING) AS LastPrice
FROM Production.ProductListPriceHistory;

Commands completed successfully.

(395 rows affected)

Total execution time: 00:00:00.052

ProductID,ListPrice,StartDate,FirstPrice,LastPrice
707,34.99,2013-05-30 00:00:00.000,33.6442,34.99
707,33.6442,2012-05-30 00:00:00.000,33.6442,34.99
707,33.6442,2011-05-31 00:00:00.000,33.6442,34.99
708,34.99,2013-05-30 00:00:00.000,33.6442,34.99
708,33.6442,2012-05-30 00:00:00.000,33.6442,34.99
708,33.6442,2011-05-31 00:00:00.000,33.6442,34.99
709,9.50,2011-05-31 00:00:00.000,9.50,9.50
710,9.50,2011-05-31 00:00:00.000,9.50,9.50
711,34.99,2013-05-30 00:00:00.000,33.6442,34.99
711,33.6442,2012-05-30 00:00:00.000,33.6442,34.99


In [14]:
--7 : Pivot order count by year 
USE AdventureWorks2019
GO
SELECT *
FROM (
  SELECT 
    YEAR(OrderDate) AS OrderYear,
    TerritoryID
  FROM Sales.SalesOrderHeader
) AS src
PIVOT (
  COUNT(TerritoryID) FOR OrderYear IN ([2011], [2012], [2013], [2014])
) AS p;

Commands completed successfully.

(1 row affected)

Total execution time: 00:00:00.070

2011,2012,2013,2014
1607,3915,14182,11761


In [16]:
--8: Get order count for every combination of year and customer
USE AdventureWorks2019
GO
SELECT 
  YEAR(OrderDate) AS OrderYear,
  CustomerID,
  COUNT(*) AS OrderCount,
  GROUPING_ID(YEAR(OrderDate), CustomerID) AS GroupTag
FROM Sales.SalesOrderHeader
GROUP BY CUBE(YEAR(OrderDate), CustomerID);

Commands completed successfully.

(45141 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:00.264

OrderYear,CustomerID,OrderCount,GroupTag
2011,11000,1,0
2013,11000,2,0
NULL,11000,3,2
2011,11001,1,0
2013,11001,1,0
2014,11001,1,0
NULL,11001,3,2
2011,11002,1,0
2013,11002,2,0
NULL,11002,3,2


In [17]:
--9: Show where each order stands among all in percent

USE AdventureWorks2019
GO
SELECT 
  SalesOrderID,
  TotalDue,
  PERCENT_RANK() OVER(ORDER BY TotalDue) AS PercentRank
FROM Sales.SalesOrderHeader;

Commands completed successfully.

(31465 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:00.221

SalesOrderID,TotalDue,PercentRank
51782,1.5183,0
51885,2.5305,3.1782354436816676E-05
51886,2.5305,3.1782354436816676E-05
52031,2.5305,3.1782354436816676E-05
52371,2.5305,3.1782354436816676E-05
52532,2.5305,3.1782354436816676E-05
52682,2.5305,3.1782354436816676E-05
52786,2.5305,3.1782354436816676E-05
53015,2.5305,3.1782354436816676E-05
53084,2.5305,3.1782354436816676E-05


In [18]:
--10: Show daily, monthly, and yearly total order quantity
USE AdventureWorks2019
GO
SELECT 
  YEAR(OrderDate) AS OrderYear,
  MONTH(OrderDate) AS OrderMonth,
  DAY(OrderDate) AS OrderDay,
  COUNT(*) AS Orders
FROM Sales.SalesOrderHeader
GROUP BY ROLLUP(YEAR(OrderDate), MONTH(OrderDate), DAY(OrderDate));


Commands completed successfully.

(1167 rows affected)

Total execution time: 00:00:00.071

OrderYear,OrderMonth,OrderDay,Orders
2011,5,31,43
2011,5,NULL,43
2011,6,1,4
2011,6,2,5
2011,6,3,2
2011,6,4,5
2011,6,5,4
2011,6,6,3
2011,6,7,3
2011,6,8,6
